# Medica: training and evaluation

Runs the repository's own `train.py` and `evaluate.py` on the held-out split. Attach the `dataset/` folder as a Kaggle Dataset, then run. See `readme.md` for the architecture and the measured scores.

In [ ]:
# --- Setup: get the repo into the notebook runtime -----------------------
# On Kaggle: add the repo as a "utility" input dataset, or set REPO_URL.
# The clone lands in /kaggle/working, which is the only writable area and
# what gets saved with the version.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/NickLitharis/medica.git"
WORK = Path("/kaggle/working")
REPO = WORK / "medica"

if not REPO.exists():
    try:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
    except Exception as e:
        print("git clone failed:", e)
        print("Upload the repo as a Kaggle dataset and set REPO to that folder instead.")
    else:
        print("Cloned to", REPO)

if REPO.exists():
    sys.path.insert(0, str(REPO))
    os.chdir(REPO)
    print("Working directory:", os.getcwd())
else:
    print("Repo missing: the rest of this notebook cannot run.")


In [ ]:
# --- Resolve input and output paths --------------------------------------
# Attach the datasets first (right sidebar -> Add Data):
#   * figure images  (PNG files, folder name does not matter)
#   * s2_cleaned.json (cleaned captions JSONL)
from config.paths import resolve, check
paths = resolve()
check(paths)


In [ ]:
# --- Stage 4: train -----------------------------------------------------
# Calls the repository's own training script (TensorFlow/Keras 2 pipeline).
# --out is where model_best.keras and encoding.json land.
!python3 train.py --dataset-dir /kaggle/input/dataset     --out /kaggle/working/medica/run --epochs 30 --batch-size 128 --arch seq2seq


In [ ]:
# --- Stage 5: score on the held-out split -------------------------------
# evaluate.py reads model_best.keras + encoding.json from --run and prints
# BLEU-1..4 against the published ImageCLEF 2017 baselines.
!python3 evaluate.py --dataset-dir /kaggle/input/dataset     --run /kaggle/working/medica/run --beam-width 1


In [ ]:
# --- Stage 6: a few generated captions, side by side --------------------
import json
from pathlib import Path
import tensorflow as tf
import pandas as pd

run = Path("/kaggle/working/medica/run")
enc = json.load(open(run / "encoding.json"))
print("run configuration:", {k: enc[k] for k in list(enc)[:8]})
print("artifacts:", sorted(p.name for p in run.iterdir()))
